# Structured Output

**Goal:** Get reliable JSON out of a model — and see exactly where and how it breaks.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [ ]:
%pip install -q groq

In [ ]:
import os

# In Colab, read the key from Secrets (key icon in the left sidebar).
# Locally, set the GROQ_API_KEY env var instead.
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except ImportError:
    assert os.environ.get('GROQ_API_KEY'), 'Set GROQ_API_KEY'

from groq import Groq
client = Groq()
MODEL = 'openai/gpt-oss-120b'  # update if you get a 404 — see 00-setup/00-environment.ipynb to list available models

## The problem

Your program needs `{"name": ..., "priority": ...}`. The model produces prose. Every production LLM feature that feeds a database, an API call, or another system hits this gap, and the naive fix — "just ask for JSON" — fails in three recurring ways:

- **Markdown fences.** The model wraps output in ` ```json ... ``` ` and `json.loads` chokes.
- **Trailing commentary.** "Here's the JSON you asked for:" before, "Let me know if..." after.
- **Schema drift.** Field names paraphrased, enums reworded, numbers returned as strings.

We'll climb a ladder of increasingly robust fixes. The test case throughout: extracting a support ticket from a free-text customer email.


In [ ]:
EMAIL = (
    'Hi team — our checkout page has been throwing 500s since about 2pm. '
    'This is blocking all purchases, we are losing money every minute. '
    'Account: acme-corp, plan: enterprise. Please treat as urgent. — Dana'
)

# The naive approach: just ask.
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket from this email as JSON with fields '
                   f'summary, account, plan, priority (low/medium/high/urgent):\n\n{EMAIL}',
    }],
)
raw = response.choices[0].message.content
print(raw)


Run it a few times. Some runs give clean JSON; others wrap it in fences or add a sentence of preamble. "Works most of the time" is the worst failure mode — it survives your demo and dies in production at 2am.

## Rung 1: prompt hard + defensive parsing

Tighten the prompt ("output ONLY the JSON object, no markdown, no commentary") and wrap the parse in error handling that strips the common wrappers. This is cheap and catches most of it — keep a helper like this around even when you use the stronger techniques below, because it costs nothing.


In [ ]:
import json
import re


def parse_json_loose(text):
    """json.loads with the two most common failure modes handled."""
    text = text.strip()
    # Strip markdown fences if present
    fence = re.match(r'^```(?:json)?\s*(.*?)\s*```$', text, re.DOTALL)
    if fence:
        text = fence.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Last resort: grab the outermost {...} span (drops pre/post commentary)
        start, end = text.find('{'), text.rfind('}')
        if start != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket from this email. Output ONLY a JSON object — '
                   f'no markdown fences, no commentary. Fields: summary (string), '
                   f'account (string), plan (string), priority (one of: low, medium, high, urgent).'
                   f'\n\n{EMAIL}',
    }],
)
ticket = parse_json_loose(response.choices[0].message.content)
print(json.dumps(ticket, indent=2))


Better, but you're still trusting the model's formatting discipline, and nothing checks the *contents* — `priority: "URGENT!!"` parses fine and breaks your enum downstream.

## Rung 2: JSON mode

Many OpenAI-compatible APIs (including Groq) support `response_format={"type": "json_object"}`. This forces the response to be valid JSON — no fences, no preamble — but doesn't validate the *shape*. Think of it as a parser guarantee, not a schema guarantee.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    response_format={"type": "json_object"},
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket as JSON with fields summary, account, '
                   f'plan, priority (low/medium/high/urgent):\n\n{EMAIL}',
    }],
)
ticket = json.loads(response.choices[0].message.content)
print(json.dumps(ticket, indent=2))


JSON mode guarantees parseable output, but the model can still drift on field names or enum values. Run it and note the shape looks right — but try changing the email to mention a "platinum plan" and re-run a few times to see enum drift.

## Rung 3: tool use as a schema enforcer

This is the production answer. Define a tool whose parameters schema *is* your output schema, force the model to call it with `tool_choice`, and read the arguments off the tool call. Three things you get for free:

- The response arrives as an **already-parsed dict** — no string parsing at all.
- You can validate *contents* (enum membership, field lengths) against actual logic.
- The schema is code, versioned next to the code that consumes it — not prose buried in a prompt.

The model never "runs" this tool; we're borrowing the tool-calling machinery purely as a typed output channel.


In [ ]:
TICKET_TOOL = {
    'type': 'function',
    'function': {
        'name': 'record_ticket',
        'description': 'Record a structured support ticket extracted from a customer email.',
        'parameters': {
            'type': 'object',
            'properties': {
                'summary':  {'type': 'string', 'description': 'One-line issue summary'},
                'account':  {'type': 'string'},
                'plan':     {'type': 'string', 'enum': ['free', 'pro', 'enterprise']},
                'priority': {'type': 'string', 'enum': ['low', 'medium', 'high', 'urgent']},
            },
            'required': ['summary', 'account', 'plan', 'priority'],
        },
    },
}

response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    tools=[TICKET_TOOL],
    tool_choice={'type': 'function', 'function': {'name': 'record_ticket'}},  # force the call
    messages=[{'role': 'user', 'content': f'Extract the support ticket:\n\n{EMAIL}'}],
)

tool_call = response.choices[0].message.tool_calls[0]
ticket = json.loads(tool_call.function.arguments)  # already valid JSON
print(json.dumps(ticket, indent=2))


## Retry on validation failure

`strict: true` covers what JSON Schema can express. Business rules it can't express (cross-field constraints, values that must exist in your database) still need your own validation — and when it fails, the right move is usually one retry that *tells the model what was wrong*. Feed the error back as a message and ask again.


In [ ]:
def validate_ticket(t):
    """Return a list of problems; empty list means valid."""
    problems = []
    if not isinstance(t.get('summary'), str) or not (5 <= len(t['summary']) <= 120):
        problems.append('summary must be a string of 5-120 characters')
    if t.get('priority') not in ('low', 'medium', 'high', 'urgent'):
        problems.append(f"priority {t.get('priority')!r} not in allowed enum")
    if t.get('plan') not in ('free', 'pro', 'enterprise'):
        problems.append(f"plan {t.get('plan')!r} not in allowed enum")
    return problems


def extract_with_retry(email, max_attempts=3):
    messages = [{'role': 'user', 'content': f'Extract the support ticket:\n\n{email}'}]
    for attempt in range(1, max_attempts + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=300,
            tools=[TICKET_TOOL],
            tool_choice={'type': 'function', 'function': {'name': 'record_ticket'}},
            messages=messages,
        )
        tool_call = response.choices[0].message.tool_calls[0]
        ticket = json.loads(tool_call.function.arguments)
        problems = validate_ticket(ticket)
        if not problems:
            print(f'valid on attempt {attempt}')
            return ticket
        print(f'attempt {attempt} failed validation: {problems}')
        # Feed the assistant message + tool result back so the model can correct itself
        messages.append({'role': 'assistant', 'content': None,
                         'tool_calls': response.choices[0].message.tool_calls})
        messages.append({'role': 'tool',
                         'tool_call_id': tool_call.id,
                         'content': 'Validation failed: ' + '; '.join(problems) +
                                    '. Call the tool again with corrected values.'})
    raise ValueError(f'no valid extraction after {max_attempts} attempts')


ticket = extract_with_retry(EMAIL)
print(json.dumps(ticket, indent=2))


Run it — this should pass on attempt 1. The retry loop is your safety net for the rules the schema can't see.

## Failure modes to keep in your head

Even with tool-forcing, watch for these in the wild:

- **Enum drift.** Without forced tool calls, a `priority` enum of `low/medium/high/urgent` will occasionally come back as `"critical"` or `"P1"`. Tool calling with a proper schema helps a lot; validate enum membership in your own code as a backstop.
- **Optional-field hallucination.** Add an optional `phone` field and the model will sometimes invent one rather than omit it — models are biased toward filling slots. Prefer required fields with explicit `null`-like sentinels, or instruct "omit fields not present in the source" and spot-check.
- **Numbers as strings.** `"amount": "42.50"` instead of `42.50`, especially when the source text contains currency symbols. Coerce and log rather than trusting the type.

JSON mode + tool forcing together: use JSON mode (`response_format`) when you own the whole schema in the prompt; use tool forcing when you want the schema as code (versioned, typeable, composable with other tools). The next notebook goes deep on the tool-calling protocol.


## Exercises

1. Switch to `response_format={"type": "json_object"}` (rung 2) and change the `plan` enum in the email to something off-list (e.g. "platinum plan"). Run 5 times and count how often the enum drifts vs rung 3 with tool forcing.
2. Extend the tool schema with an optional `affected_since` field (ISO timestamp). Feed it emails that do and don't mention a time, and measure how often the model invents one.
3. Build `extract_batch(emails)` that runs the rung-3 extractor over a list of 5 short emails (loop over them sequentially), collecting results and validation failures separately.
4. Replace `validate_ticket` with a `jsonschema.validate()` call (the `jsonschema` package ships with Colab) against the same schema dict, and adapt the retry loop to surface its error messages to the model.
